In [ ]:
import boto3
import json
import time
import pandas as pd
import argparse 
import random

session = boto3.Session(profile_name='corp-us-east-1')
# session = boto3.Session(profile_name='default')


# Define the table name
table_name = 'prompt_hub_table'
# Create DynamoDB resource
dynamodb = session.resource('dynamodb')
table = dynamodb.Table(table_name)

In [ ]:

def filter_items(field,val):
    # Scan the table to get all items where is_external is true
    response = table.scan(
        FilterExpression=f'{field} = :val',
        ExpressionAttributeValues={':val': val}
    )
    
    items = response['Items']
    print(f"found:{len(items)}")
    
    # Handle pagination if there are more items
    while 'LastEvaluatedKey' in response:
        response = table.scan(
            FilterExpression='{field} = :val',
            ExpressionAttributeValues={':val': val},
            ExclusiveStartKey=response['LastEvaluatedKey']
        )
        items.extend(response['Items'])
    return items

def update_items(items,field,val):
    # Update each item's delete_status
    updated_count = 0
    for item in items:
        # Get the primary key values from your item
        # Modify these according to your table's primary key structure
        key = {
            'id': item['id']  # Assuming 'id' is your primary key
            # Add other key attributes if you have a composite key
        }

        # Update the item
        table.update_item(
            Key=key,
            UpdateExpression=f'SET {field} = :val',
            ExpressionAttributeValues={
                ':val': val
            }
        )
        updated_count += 1

    print(f"Successfully updated {updated_count} items")
    return updated_count

def delete_items(items):
    deleted_count = 0
    
    for item in items:
        try:
            # Get the primary key values from your item
            key = {
                'id': item['id']  # Assuming 'id' is your primary key
                # Add other key attributes if you have a composite key
            }

            # Delete the item
            table.delete_item(
                Key=key
            )
            
            deleted_count += 1
            
        except Exception as e:
            print(f"Error deleting item with id {item['id']}: {str(e)}")
            continue

    print(f"Successfully deleted {deleted_count} items")
    return deleted_count

In [ ]:
# # 把is_external = true都删除
# items = filter_items("is_external",True)
# update_items(items, "delete_status","deleted")

In [ ]:
# 删除之前的老版本
# items = filter_items("demo_version","2025v1")
# delete_items(items)

In [ ]:
# 把is_recommended = true 改成 is_recommended = False
# items = filter_items("is_recommended",False)
# update_items(items, "is_recommended",True)

## upload new template

In [ ]:


def generate_id():
    timestamp = int(time.time() * 1000)  # Get the current timestamp in milliseconds
    random_number = str(random.randint(0, 16**6))  # Generate a random 6-digit number
    return f"{timestamp}-{random_number}"


In [ ]:
import hashlib
from botocore.exceptions import ClientError

def calculate_record_hash(record):
    """
    计算记录哈希值(忽略id、createtime和update_time)
    """
    record_copy = record.copy()
    
    # 移除不需要参与哈希计算的字段
    for field in ['id', 'createtime', 'update_time']:
        if field in record_copy:
            del record_copy[field]
    
    # 对字典进行排序，确保相同内容产生相同哈希
    record_json = json.dumps(record_copy, sort_keys=True)
    return hashlib.md5(record_json.encode()).hexdigest()

def get_id_map_from_dynamodb(table_name):
    """
    从DynamoDB表中获取demo_name到id的映射
    """
    dynamodb = session.resource('dynamodb')
    table = dynamodb.Table(table_name)
    
    response = table.scan()
    id_map = {item['demo_name']: item['id'] for item in response['Items'] 
              if 'demo_name' in item and 'id' in item}
    
    # 处理可能的分页结果
    while 'LastEvaluatedKey' in response:
        response = table.scan(ExclusiveStartKey=response['LastEvaluatedKey'])
        for item in response['Items']:
            if 'demo_name' in item and 'id' in item:
                id_map[item['demo_name']] = item['id']
    
    return id_map

def process_excel(filename, id_map=None):
    df = pd.read_excel(filename)
    time_tuple = time.localtime(time.time())
    createtime = time.strftime("%Y-%m-%d %H:%M:%S", time_tuple)
    df.dropna(inplace=True)
    df.rename(columns={'Scenario':'category',
                       'Name':'demo_name',
                       'Description':'description',
                       'Further Support':'further_support',
                       'Status':'demo_type',
                       'Simple Demo Introduction Deck':'deck_link',
                       'Demo Video Link':'demo_link',
                       'Code Repo':'code_repo_link',
                       'China Region Support':'china_region_support',
                       'Contact':'contact',
                       'Team':'team',
                       'Industries':'industry'
                       }, inplace=True)
    
    # 使用id_map给已存在记录分配ID，或为新记录生成ID
    if id_map is not None:
        df['id'] = df['demo_name'].apply(lambda x: id_map.get(x, generate_id()))
    else:
        df['id'] = df.apply(lambda x: generate_id(), axis=1)
        
    df['createtime'] = createtime
    df['update_time'] = createtime  # 初始update_time与createtime相同
    df['company'] = 'default'
    df['template'] = ''
    df['demo_version'] = '2025v1'
    df['industry'] = df.apply(lambda x: [ i.strip()  for i in x['industry'].split(',') ] if isinstance(x['industry'], str) else [], axis=1)
    
    df_dict = json.loads(df.to_json(orient='index'))
    return list(df_dict.values())

def upload_to_dynamodb(table_name, json_data):
    dynamodb = session.resource('dynamodb')
    table = dynamodb.Table(table_name)
    
    uploaded_count = 0
    skipped_count = 0
    
    for item in json_data:
        # 查询DynamoDB中的现有记录
        try:
            response = table.get_item(Key={'id': item['id']})
        except ClientError as e:
            print(f"Error retrieving item: {e.response['Error']['Message']}")
            continue
        
        # 计算当前记录的哈希值
        current_hash = calculate_record_hash(item)
        
        # 检查记录是否存在及其哈希值
        if 'Item' in response:
            existing_item = response['Item']
            existing_hash = calculate_record_hash(existing_item)
            
            # 如果哈希值相同，内容未变化，跳过
            if current_hash == existing_hash:
                print(f"Skipping unchanged item: {item['demo_name']}")
                skipped_count += 1
                continue
                
            # 哈希值不同，更新记录并更新update_time
            time_tuple = time.localtime(time.time())
            item['update_time'] = time.strftime("%Y-%m-%d %H:%M:%S", time_tuple)
            # 保留原始的createtime
            item['createtime'] = existing_item['createtime']
            
            table.put_item(Item=item)
            print(f"Updated item: {item['demo_name']}")
            uploaded_count += 1
        else:
            # 新记录，直接添加
            table.put_item(Item=item)
            print(f"Created new item: {item['demo_name']}")
            uploaded_count += 1
    
    print(f"Data uploaded to DynamoDB table: {table_name}")
    print(f"Items uploaded: {uploaded_count}, Items skipped: {skipped_count}")


In [ ]:


excel_file = "GCR GenAI Asset Hub Management_0321.xlsx"

# 1. 获取现有记录的ID映射
id_map = get_id_map_from_dynamodb(table_name)
id_map

In [ ]:
df = pd.read_excel(excel_file)
time_tuple = time.localtime(time.time())
createtime = time.strftime("%Y-%m-%d %H:%M:%S", time_tuple)
df.dropna(inplace=True)
df.rename(columns={'Scenario':'category',
                    'Name':'demo_name',
                    'Description':'description',
                    'Further Support':'further_support',
                    'Status':'demo_type',
                    'Simple Demo Introduction Deck':'deck_link',
                    'Demo Video Link':'demo_link',
                    'Code Repo':'code_repo_link',
                    'China Region Support':'china_region_support',
                    'Contact':'contact',
                    'Team':'team',
                    'Industries':'industry'
                    }, inplace=True)
print(df)
# 使用id_map给已存在记录分配ID，或为新记录生成ID
if id_map is not None:
    df['id'] = df['demo_name'].apply(lambda x: id_map.get(x, generate_id()))
else:
    df['id'] = df.apply(lambda x: generate_id(), axis=1)
    
df['createtime'] = createtime
df['update_time'] = createtime  # 初始update_time与createtime相同
df['company'] = 'default'
df['template'] = ''
df['demo_version'] = '2025v1'
df['industry'] = df.apply(lambda x: [ i.strip()  for i in x['industry'].split(',') ] if isinstance(x['industry'], str) else [], axis=1)

df_dict = json.loads(df.to_json(orient='index'))

In [ ]:

# 2. 处理Excel文件
json_data = process_excel(excel_file, id_map)
len(json_data)

In [ ]:

# 3. 上传到DynamoDB，带校验
upload_to_dynamodb(table_name, json_data)